# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

Unit of Analysis: One row represents the daily search and engagement performance for a single unique page (content_hash_id) for a specific client (client_hash_id) on a specific day (report_date).

Time Window: A mid-panel historical month, specifically March 2026 (2026-03-01 to 2026-03-31), which allows us to leave the final month (June) untouched as a sealed test set.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Features: impressions, clicks, position, content_age_days (observable signals known at the decision moment).

Label / Proxy: is_declining_label (a future-looking proxy determining if traffic drops in the subsequent window).

Context: client_hash_id, content_hash_id, report_date (used for grouping, joining, and verifying grain; never used as ML features).

Excluded: Rows where ga4_data_available IS FALSE or gsc_data_available IS FALSE.

    Why excluded: If tracking hasn't started or is broken, the metrics show up as zeroes. Treating a tracking error as "zero traffic" will poison the model's training data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb

# ⚠️ PASTE YOUR TOKEN HERE, BUT DELETE IT BEFORE YOU GIT COMMIT!
HF_TOKEN = ''

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

# Target the daily facts table directly from Hugging Face
fact_table = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet"

# 1. Verify the Grain: Is one row truly unique per day + client + content?
print("1. GRAIN CHECK (Should find 0 duplicates):")
grain_df = con.execute(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as duplicates
    FROM read_parquet('{fact_table}')
    WHERE report_date = '2026-03-15'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()
print(f"Duplicates found: {len(grain_df)}\n")

# 2. Verify Windows & Counts: Checking our March 2026 slice
print("2. DATE WINDOW & ROW COUNT:")
span_df = con.execute(f"""
    SELECT MIN(report_date) as start_date, MAX(report_date) as end_date, COUNT(*) as total_rows
    FROM read_parquet('{fact_table}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()
display(span_df)

# 3. Verify Missing Values / Availability (Using IS TRUE)
print("\n3. DATA AVAILABILITY (Active Tracking):")
avail_df = con.execute(f"""
    SELECT COUNT(*) as valid_tracking_rows
    FROM read_parquet('{fact_table}')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
      AND ga4_data_available IS TRUE 
      AND gsc_data_available IS TRUE
""").df()
display(avail_df)

1. GRAIN CHECK (Should find 0 duplicates):
Duplicates found: 0

2. DATE WINDOW & ROW COUNT:


,start_date,end_date,total_rows
0,2026-03-01,2026-03-31,9841378



3. DATA AVAILABILITY (Active Tracking):


,valid_tracking_rows
0,364347


## 4. Data limits

By this data We got to know that :

    Unbalanced Tracking History: As proven by my query, out of 9.8 million rows in March 2026, only ~364,000 had both GA4 and GSC tracking fully active. This means a massive portion of the historical data is GSC-only. We cannot reliably analyze on-page engagement (sessions, bounce rates) across the entire dataset without severely restricting our sample size.

    Causality vs. Seasonality: The data is strictly observational. It can tell us a page lost traffic, but it can never definitively prove if the drop was due to structural content decay or just a natural seasonal dip in search demand. It also cannot prove that an editorial refresh actually caused a subsequent recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

[x] Every section above is filled — markdown thinking AND the code that backs it

[x] The notebook runs top to bottom with no errors (Runtime → Run all)

[x] No client names, URLs, or private queries anywhere

[x] My claims use careful words: observed, measured, directional, decision-support

[x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.